In [1]:
import numpy as np
import pandas as pd
import sklearn 
import tensorflow as tf
import matplotlib.pyplot as plt

In [2]:
labels = np.array(["cat", "dog", "mouse", "cat", "dog", "cat", "cat"]).reshape(-1, 1)
records, _ = labels.shape
labels, records

(array([['cat'],
        ['dog'],
        ['mouse'],
        ['cat'],
        ['dog'],
        ['cat'],
        ['cat']], dtype='<U5'),
 7)

### One Hot Encoding

In [3]:
# Library

from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(sparse_output=False)
onehot = encoder.fit_transform(labels)
onehot

array([[1., 0., 0.],
       [0., 1., 0.],
       [0., 0., 1.],
       [1., 0., 0.],
       [0., 1., 0.],
       [1., 0., 0.],
       [1., 0., 0.]])

In [4]:
# Scratch

vocab = list(sorted(set(labels.flatten().copy())))
vocab_size = len(vocab)
onehot = np.array([ [ 0 for _ in range(vocab_size) ] for _ in range(records) ])

for i in range(records):
    label = labels[i, 0]
    index = vocab.index(label)
    onehot[i, index] += 1

onehot

array([[1, 0, 0],
       [0, 1, 0],
       [0, 0, 1],
       [1, 0, 0],
       [0, 1, 0],
       [1, 0, 0],
       [1, 0, 0]])

### Bag of Words

In [5]:
# Scratch

statements = [
    "cat mat sat",
    "rat sat dog mat",
    "cat dog sat"
]

vocab = set( word for stat in statements for word in stat.split(' ')  )
vocab = sorted(vocab)

vocab_size, max_length = len(vocab), max(len(stat) for stat in statements)

bow = []
for stat in statements:
    code = [0] * vocab_size
    for word in stat.split(' '):
        index = vocab.index(word)
        code[index] = 1
    bow.append(code)
    
np.array(bow)

array([[1, 0, 1, 0, 1],
       [0, 1, 1, 1, 1],
       [1, 1, 0, 0, 1]])

In [6]:
# Library
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer()
ans = vectorizer.fit_transform(statements)
ans.toarray()

array([[1, 0, 1, 0, 1],
       [0, 1, 1, 1, 1],
       [1, 1, 0, 0, 1]])

### Preprocessing Text Pipeline

In [13]:
import re, nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer
from nltk.stem import WordNetLemmatizer

def lower(text):
    return text.lower()

def punct_remove(text):
    text = re.sub(r'[^\w\s]', '', text)
    text = re.sub(r'http?://\S+', '', text)
    return text 

def tokenize(text):
    return text.split(' ')

def stop_words(tokens):
    _stopwords = set(stopwords.words('english'))
    return [ token for token in tokens if token not in _stopwords and token.isalpha() ]

def stemming(tokens):
    stemmer = PorterStemmer()
    return [ stemmer.stem(token) for token in tokens ]

def lemmatizing(tokens):
    lemmatizer = WordNetLemmatizer()
    return [ lemmatizer.lemmatize(token) for token in tokens ]
    
def preprocess_text(text):
    text = lower(text)
    text = punct_remove(text)
    tokens = tokenize(text)
    tokens = stop_words(tokens)
    tokens = stemming(tokens)
    tokens = lemmatizing(tokens)
    return ' '.join(tokens)


text = "This is unprocessed text 23i with the different, punctuations and    tokens for testing    teh preprocessing pipeline"
preprocess_text(text)

'unprocess text differ punctuat token test teh preprocess pipelin'

### N-grams

In [19]:
corpus = [
    "nlp is fun and interactive",
    "machine learning is a part of nlp",
    "interactive machine learning is fun",
]

n = 3
ngrams = []
for text in corpus:
    text = ["<s>"] * (n-1) + text.split(' ') + ["</s>"] * (n-1)
    for i in range(len(text)-n+1):
        ngrams.append(text[i:i+n])

ngrams

[['<s>', '<s>', 'nlp'],
 ['<s>', 'nlp', 'is'],
 ['nlp', 'is', 'fun'],
 ['is', 'fun', 'and'],
 ['fun', 'and', 'interactive'],
 ['and', 'interactive', '</s>'],
 ['interactive', '</s>', '</s>'],
 ['<s>', '<s>', 'machine'],
 ['<s>', 'machine', 'learning'],
 ['machine', 'learning', 'is'],
 ['learning', 'is', 'a'],
 ['is', 'a', 'part'],
 ['a', 'part', 'of'],
 ['part', 'of', 'nlp'],
 ['of', 'nlp', '</s>'],
 ['nlp', '</s>', '</s>'],
 ['<s>', '<s>', 'interactive'],
 ['<s>', 'interactive', 'machine'],
 ['interactive', 'machine', 'learning'],
 ['machine', 'learning', 'is'],
 ['learning', 'is', 'fun'],
 ['is', 'fun', '</s>'],
 ['fun', '</s>', '</s>']]

### Naive Bayes Classifier

In [28]:
corpus = ["I love this class", "Brilliant lecture", "Horrible assignment", "I hate this task"]
labels = [1, 1, 0, 0]

# Library
from sklearn.naive_bayes import MultinomialNB
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import classification_report

vectorizer = CountVectorizer()
X = vectorizer.fit_transform(corpus).toarray()
y = np.array(labels)

nb = MultinomialNB()
nb.fit(X, y)
predictions = nb.predict(X)

print(classification_report(y, predictions))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00         2
           1       1.00      1.00      1.00         2

    accuracy                           1.00         4
   macro avg       1.00      1.00      1.00         4
weighted avg       1.00      1.00      1.00         4



### TF-IDF

In [30]:
# Library
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf = TfidfVectorizer()
tfidf.fit_transform(corpus).toarray()

array([[0.        , 0.        , 0.61761437, 0.        , 0.        ,
        0.        , 0.61761437, 0.        , 0.48693426],
       [0.        , 0.70710678, 0.        , 0.        , 0.        ,
        0.70710678, 0.        , 0.        , 0.        ],
       [0.70710678, 0.        , 0.        , 0.        , 0.70710678,
        0.        , 0.        , 0.        , 0.        ],
       [0.        , 0.        , 0.        , 0.61761437, 0.        ,
        0.        , 0.        , 0.61761437, 0.48693426]])

In [50]:
# Scratch

# 1 - TF: Term Frequency
tf = []
for sent in bow:
    temp = sent.copy()
    tf.append([ t / len(sent) for t in temp ])
bow, tf

# 2 - IDF: Inverse Document Frequency
frequencies = {}
for stat in statements:
    for word in stat.split(' '):
        if word in frequencies:
            frequencies[word] += 1
        else:
            frequencies[word] = 1

frequencies

# IDF = log( 1+statements / word in document )
idf = {}
for key, value in frequencies.items():
    idf[key] = np.log( (len(statements)+1) / frequencies[key] )

tfidf = []
for i in range(len(statements)):
    row = []
    for j in range(vocab_size):
        word = vocab[j]
        row.append( tf[i][j] * idf.get(word, 0) )
    tfidf.append(row)

np.array(tfidf)

# A new sentence to test
new_sentence = "cat mat sat"

# Step A: Make a BOW count for the new sentence using your vocabulary
new_bow = []
for word in vocab:
    count = new_sentence.split().count(word)
    new_bow.append(count)

print("\nNew Sentence BOW:", new_bow)

# Step B: Calculate TF for the new sentence
total_new_words = sum(new_bow)
if total_new_words > 0:
    new_tf = [count / total_new_words for count in new_bow]
else:
    new_tf = [0 for _ in new_bow]

# Step C: Calculate final TF-IDF for the new sentence
new_tfidf = []
for j in range(len(vocab)):
    word = vocab[j]
    term_freq = new_tf[j]
    # Use the idf we calculated earlier from the training data
    word_idf = idf.get(word, 0)
    new_tfidf.append(term_freq * word_idf)

print("New Sentence TF-IDF Vector:", new_tfidf)


New Sentence BOW: [1, 0, 1, 0, 1]
New Sentence TF-IDF Vector: [np.float64(0.23104906018664842), np.float64(0.0), np.float64(0.23104906018664842), np.float64(0.0), np.float64(0.09589402415059362)]


### HMM - Hidden Markov Model

In [99]:
# Library 
from collections import Counter, defaultdict
import pandas as pd

s1 = [
    [(" ", "marry", "jane", "can", "see", "will", " "), ("<s>", "N", "N", "M", "V", "N", "<e>")],
    [(" ", "will", "jane", "spot", "marry", " "), ("<s>", "M", "N", "V", "N", "<e>")],
    [(" ", "spot", "will", "see", "marry", " "), ("<s>", "N", "M", "V", "N", "<e>")],
    [(" ", "marry", "will", "pat", "spot", " "), ("<s>", "N", "M", "V", "N", "<e>")]
]

tags, words = [], []
for word, tag in s1:
    tags.extend(tag)
    words.extend(word)

print("Tags ", tags, "\nWords ", words)

# Emission = { tag: {word: freqs} }
# Transition = {tag: {nexttag: freqs} }

emission_counts = defaultdict(lambda: defaultdict(int))
transition_counts = defaultdict(lambda: defaultdict(int))
for word, tag in zip(words, tags):
    emission_counts[tag][word] += 1

for _, tgs in s1:
    for i in range(len(tgs)-1):
        transition_counts[tgs[i]][tgs[i+1]] += 1

# emission_counts
transition_counts

Tags  ['<s>', 'N', 'N', 'M', 'V', 'N', '<e>', '<s>', 'M', 'N', 'V', 'N', '<e>', '<s>', 'N', 'M', 'V', 'N', '<e>', '<s>', 'N', 'M', 'V', 'N', '<e>'] 
Words  [' ', 'marry', 'jane', 'can', 'see', 'will', ' ', ' ', 'will', 'jane', 'spot', 'marry', ' ', ' ', 'spot', 'will', 'see', 'marry', ' ', ' ', 'marry', 'will', 'pat', 'spot', ' ']


defaultdict(<function __main__.<lambda>()>,
            {'<s>': defaultdict(int, {'N': 3, 'M': 1}),
             'N': defaultdict(int, {'N': 1, 'M': 3, '<e>': 4, 'V': 1}),
             'M': defaultdict(int, {'V': 3, 'N': 1}),
             'V': defaultdict(int, {'N': 4})})

In [101]:
emission_prob = defaultdict(lambda: defaultdict(int))
transition_prob = defaultdict(lambda: defaultdict(int))

for tag in emission_counts:
    total = sum(emission_counts[tag].values())
    if tag != "<s>" and tag != "<e>":
        for word in emission_counts[tag]:
            emission_prob[tag][word] = emission_counts[tag][word] / total

for tag in transition_counts:
    total = sum(transition_counts[tag].values())
    for nw in transition_counts[tag]:
        transition_prob[tag][nw] = transition_counts[tag][nw] / total

transition_prob

defaultdict(<function __main__.<lambda>()>,
            {'<s>': defaultdict(int, {'N': 0.75, 'M': 0.25, ' ': 0.0}),
             'N': defaultdict(int,
                         {'N': 0.1111111111111111,
                          'M': 0.3333333333333333,
                          '<e>': 0.4444444444444444,
                          'V': 0.1111111111111111,
                          'marry': 0.0,
                          'jane': 0.0,
                          'will': 0.0,
                          'spot': 0.0}),
             'M': defaultdict(int,
                         {'V': 0.75, 'N': 0.25, 'can': 0.0, 'will': 0.0}),
             'V': defaultdict(int,
                         {'N': 1.0, 'see': 0.0, 'spot': 0.0, 'pat': 0.0})})